# Experiment 3: normalized category profiles

One DAMICORE object represents one eligible source category. The input documents
are fixed-width relative-context profiles created by `00_create_artifacts`.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    artifact_paths,
    case_execution,
    category_order_from_manifest,
    ensure_artifact_directories,
    load_artifact_manifest,
    load_category_map,
    run_damicore_experiment,
    write_common_result_artifacts,
)

CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
PATHS = artifact_paths(CATEGORY_SET_VERSION)
COMMON_WORK_ROOT = PATHS.common
NORMALIZED_WORK_ROOT = PATHS.normalized
CASE_FULL_WORK_ROOT = PATHS.case_full
CASE_BALANCED_WORK_ROOT = PATHS.case_balanced
RESULTS_ROOT = PATHS.results
ensure_artifact_directories(PATHS)
manifest = load_artifact_manifest(
    COMMON_WORK_ROOT / "artifact-manifest.json",
    category_set_version=CATEGORY_SET_VERSION,
)
category_order = category_order_from_manifest(manifest)
category_map = load_category_map(COMMON_WORK_ROOT / "category-map.csv")
category_support = pd.read_csv(COMMON_WORK_ROOT / "category-support.csv")
assert category_map["category"].tolist() == category_order
assert set(category_support["category"]) == set(category_order)
assert manifest["dimension_count"] == 20


## Run DAMICORE and record the common result contract


In [ ]:
result = run_damicore_experiment(
    experiment_name="normalized_categories",
    corpus_dir=NORMALIZED_WORK_ROOT / "corpus",
    raw_runs_dir=NORMALIZED_WORK_ROOT / "runs",
    execution=case_execution(),
)
assert result["status"] == "completed", result["preview"]

bytes_by_category = {
    row.category: int((NORMALIZED_WORK_ROOT / "corpus" / row.label).stat().st_size)
    for row in category_map.itertuples(index=False)
}
output_dir = RESULTS_ROOT / "normalized_categories"
normalized_result = write_common_result_artifacts(
    result=result,
    category_map=category_map,
    category_order=category_order,
    support=category_support,
    bytes_by_category=bytes_by_category,
    output_dir=output_dir,
    support_column="support",
    support_label="Distinct reports (log scale)",
    title_prefix="Normalized categories",
    manifest=manifest,
)
display(normalized_result["membership_by_category"])
display(normalized_result["distance"].round(3))


## Interpretation boundary

These clusters describe similarity among normalized contextual profiles. They are
exploratory and are not legal categories, causal mechanisms, or individual-level
predictions.
